# Baseline ARM-FM — "after" vs "only after"

Generate **only** the Reward Machine with the ARM-FM baseline generator, for matched
task pairs that differ only by the word *only*. No labeling functions, no state
descriptions, no bundle, and no training are produced.

The goal is to see whether the baseline changes the RM topology when an ordering is
stated as a plain sequence ("after") versus an exclusive precondition ("only after").


In [1]:
%load_ext autoreload
%autoreload 2

import dotenv

from maikol_utils.print_utils import print_error, print_separator

from src.arm_fm import parse_paper_reward_machine, serialize_paper_reward_machine
from src.arm_fm.generation import _extract_artifact, _request_text
from src.config import Configuration
from src.models import EnvironmentDescription
from src.utils import get_engine

dotenv.load_dotenv()


True

## Setup


In [2]:
CONFIG = Configuration()
EXAMPLES = CONFIG.WORKSPACE_PATH / "examples" / "arm-fm"


## Task pairs

Each pair uses the same environment and the same sentence, changing only `after` into
`only after`. The environments are the paper's catalogue entries under `examples/arm-fm/`.


In [3]:
TASK_PAIRS = [
    (
        "MiniGrid DoorKey",
        "minigrid-doorkey/environment.md",
        "Open the door after acquiring the key.",
        "Open the door only after acquiring the key.",
    ),
    (
        "BabyAI UnlockToUnlock",
        "babyai-unlock-to-unlock/environment.md",
        "Open the prerequisite door after acquiring its key.",
        "Open the prerequisite door only after acquiring its key.",
    ),
    (
        "Craftium Diamond",
        "craftium-diamond/environment.md",
        "Acquire iron after acquiring stone.",
        "Acquire iron only after acquiring stone.",
    ),
    (
        "Meta-World Pick-Place",
        "metaworld-pick-place/environment.md",
        "Place the puck at the goal after moving it near the goal.",
        "Place the puck at the goal only after moving it near the goal.",
    ),
]


## Generate only the Reward Machine

One baseline `rm_generator` request per task, using the baseline's own request and fence
helpers. The later baseline stages (critic, labeling, descriptions) are skipped.


In [6]:
for label, environment_file, after_task, only_after_task in TASK_PAIRS:
    environment = EnvironmentDescription.from_file(EXAMPLES / environment_file)
    engine = get_engine(CONFIG, environment)

    for wording, task in (("after", after_task), ("only after", only_after_task)):
        print_separator(f"{label} — {wording}", sep_type="NORMAL")
        print(f"TASK: {task}\n")

        try:
            candidate = _request_text(
                engine,
                "rm_generator",
                environment=environment.markdown,
                task=task,
                history="",
            )
            machine = parse_paper_reward_machine(
                _extract_artifact(candidate),
                propositions=environment.proposition_ids,
                require_final_states=True,
            )
            print(serialize_paper_reward_machine(machine))
        except Exception as error:
            print_error(f"Generation failed: {error}")


________________________________________________________________
                    MiniGrid DoorKey — after                    

TASK: Open the door after acquiring the key.

REWARD_MACHINE:
STATES: u0, u1, u2
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, has_key) -> u1
(u0, else) -> u0
(u1, door_open) -> u2
(u1, key_lost) -> u0
(u1, else) -> u1
(u2, else) -> u2
REWARD_FUNCTION:
(u0, has_key, u1) -> 0.3
(u1, door_open, u2) -> 1

________________________________________________________________
                 MiniGrid DoorKey — only after                  

TASK: Open the door only after acquiring the key.

REWARD_MACHINE:
STATES: u0, u1, u2
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, has_key) -> u1
(u0, else) -> u0
(u1, door_open) -> u2
(u1, key_lost) -> u1
(u1, else) -> u1
(u2, else) -> u2
REWARD_FUNCTION:
(u0, has_key, u1) -> 0.3
(u1, door_open, u2) -> 1
(u1, key_lost, u1) -> -0.1

________________________________________________________________
   

```
________________________________________________________________
                    MiniGrid DoorKey — after                    

TASK: Open the door after acquiring the key.

REWARD_MACHINE:
STATES: u0, u1, u2
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, has_key) -> u1
(u0, else) -> u0
(u1, door_open) -> u2
(u1, else) -> u1
(u2, else) -> u2
REWARD_FUNCTION:
(u0, has_key, u1) -> 0.3
(u1, door_open, u2) -> 0.7

________________________________________________________________
                 MiniGrid DoorKey — only after                  

TASK: Open the door only after acquiring the key.

REWARD_MACHINE:
STATES: u0, u1, u2
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, has_key) -> u1
(u0, else) -> u0
(u1, door_open) -> u2
(u1, key_lost) -> u1
(u1, else) -> u1
(u2, else) -> u2
REWARD_FUNCTION:
(u0, has_key, u1) -> 0.2
(u1, door_open, u2) -> 1
(u1, key_lost, u1) -> -0.1

________________________________________________________________
                 BabyAI UnlockToUnlock — after                  

TASK: Open the prerequisite door after acquiring its key.

REWARD_MACHINE:
STATES: u0, u1, u2
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, has_prerequisite_key) -> u1
(u0, else) -> u0
(u1, prerequisite_door_open) -> u2
(u1, required_key_lost) -> u0
(u1, else) -> u1
(u2, else) -> u2
REWARD_FUNCTION:
(u0, has_prerequisite_key, u1) -> 0.2
(u1, prerequisite_door_open, u2) -> 1

________________________________________________________________
               BabyAI UnlockToUnlock — only after               

TASK: Open the prerequisite door only after acquiring its key.

REWARD_MACHINE:
STATES: u0, u1, u2
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, has_prerequisite_key) -> u1
(u0, else) -> u0
(u1, prerequisite_door_open) -> u2
(u1, required_key_lost) -> u0
(u1, else) -> u1
(u2, else) -> u2
REWARD_FUNCTION:
(u0, has_prerequisite_key, u1) -> 0.3
(u1, prerequisite_door_open, u2) -> 1
(u1, required_key_lost, u0) -> -0.1

________________________________________________________________
                    Craftium Diamond — after                    

TASK: Acquire iron after acquiring stone.

REWARD_MACHINE:
STATES: u0, u1, u2
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, stone_acquired) -> u1
(u0, else) -> u0
(u1, iron_acquired) -> u2
(u1, else) -> u1
REWARD_FUNCTION:
(u0, stone_acquired, u1) -> 0.1
(u1, iron_acquired, u2) -> 1

________________________________________________________________
                 Craftium Diamond — only after                  

TASK: Acquire iron only after acquiring stone.

REWARD_MACHINE:
STATES: u0, u1, u2, u3
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, iron_acquired) -> u3
(u0, stone_acquired) -> u1
(u0, else) -> u0
(u1, iron_acquired) -> u2
(u1, else) -> u1
(u2, else) -> u2
(u3, else) -> u3
REWARD_FUNCTION:
(u0, stone_acquired, u1) -> 0.3
(u1, iron_acquired, u2) -> 1

________________________________________________________________
                 Meta-World Pick-Place — after                  

TASK: Place the puck at the goal after moving it near the goal.

REWARD_MACHINE:
STATES: u0, u1, u2
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, object_near_goal | object_at_goal) -> u1
(u0, else) -> u0
(u1, object_at_goal) -> u2
(u1, else) -> u1
(u2, else) -> u2
REWARD_FUNCTION:
(u0, object_near_goal | object_at_goal, u1) -> 0.5
(u1, object_at_goal, u2) -> 1

________________________________________________________________
               Meta-World Pick-Place — only after               

TASK: Place the puck at the goal only after moving it near the goal.

REWARD_MACHINE:
STATES: u0, u1, u2, u3
INITIAL_STATE: u0
FINAL_STATES: u3
TRANSITION_FUNCTION:
(u0, object_grasped) -> u1
(u0, else) -> u0
(u1, object_near_goal) -> u2
(u1, else) -> u1
(u2, object_at_goal) -> u3
(u2, else) -> u2
(u3, else) -> u3
REWARD_FUNCTION:
(u0, object_grasped, u1) -> 0.1
(u1, object_near_goal, u2) -> 0.3
(u2, object_at_goal, u3) -> 1
```

```
________________________________________________________________
                    MiniGrid DoorKey — after                    

TASK: Open the door after acquiring the key.

REWARD_MACHINE:
STATES: u0, u1, u2
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, has_key) -> u1
(u0, else) -> u0
(u1, key_lost) -> u0
(u1, has_key & door_open) -> u2
(u1, else) -> u1
(u2, else) -> u2
REWARD_FUNCTION:
(u0, has_key, u1) -> 0.3
(u1, has_key & door_open, u2) -> 1

________________________________________________________________
                 MiniGrid DoorKey — only after                  

TASK: Open the door only after acquiring the key.

REWARD_MACHINE:
STATES: u0, u1, u2
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, has_key) -> u1
(u0, else) -> u0
(u1, door_open) -> u2
(u1, else) -> u1
(u2, else) -> u2
REWARD_FUNCTION:
(u0, has_key, u1) -> 0.4
(u1, door_open, u2) -> 1

________________________________________________________________
                 BabyAI UnlockToUnlock — after                  

TASK: Open the prerequisite door after acquiring its key.

REWARD_MACHINE:
STATES: u0, u1, u2
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, has_prerequisite_key & !prerequisite_door_open) -> u1
(u0, else) -> u0
(u1, prerequisite_door_open) -> u2
(u1, else) -> u1
REWARD_FUNCTION:
(u0, has_prerequisite_key & !prerequisite_door_open, u1) -> 0.3
(u1, prerequisite_door_open, u2) -> 1

________________________________________________________________
               BabyAI UnlockToUnlock — only after               

TASK: Open the prerequisite door only after acquiring its key.

REWARD_MACHINE:
STATES: u0, u1, u2, u3
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, has_prerequisite_key) -> u1
(u0, else) -> u0
(u1, prerequisite_door_open) -> u2
(u1, required_key_lost) -> u3
(u1, else) -> u1
(u2, else) -> u2
(u3, else) -> u3
REWARD_FUNCTION:
(u0, has_prerequisite_key, u1) -> 0.3
(u1, prerequisite_door_open, u2) -> 1
(u1, required_key_lost, u3) -> -0.2

________________________________________________________________
                    Craftium Diamond — after                    

TASK: Acquire iron after acquiring stone.

REWARD_MACHINE:
STATES: u0, u1, u2
INITIAL_STATE: u0
FINAL_STATES: u2
TRANSITION_FUNCTION:
(u0, stone_acquired) -> u1
(u0, else) -> u0
(u1, iron_acquired) -> u2
(u1, else) -> u1
(u2, else) -> u2
REWARD_FUNCTION:
(u0, stone_acquired, u1) -> 0.3
(u1, iron_acquired, u2) -> 1

________________________________________________________________
                 Craftium Diamond — only after                  

TASK: Acquire iron only after acquiring stone.

REWARD_MACHINE:
STATES: u0, u1, u2, u3
INITIAL_STATE: u0
FINAL_STATES: u3
TRANSITION_FUNCTION:
(u0, stone_acquired & !iron_acquired) -> u1
(u0, iron_acquired & !stone_acquired) -> u2
(u0, else) -> u0
(u1, iron_acquired) -> u3
(u1, else) -> u1
(u2, else) -> u2
(u3, else) -> u3
REWARD_FUNCTION:
(u0, stone_acquired & !iron_acquired, u1) -> 0.4
(u1, iron_acquired, u3) -> 1

________________________________________________________________
                 Meta-World Pick-Place — after                  

TASK: Place the puck at the goal after moving it near the goal.

REWARD_MACHINE:
STATES: u0, u1, u2, u3, u4
INITIAL_STATE: u0
FINAL_STATES: u4
TRANSITION_FUNCTION:
(u0, near_object & !object_grasped) -> u1
(u0, else) -> u0
(u1, object_grasped) -> u2
(u1, else) -> u1
(u2, object_near_goal | object_at_goal) -> u3
(u2, else) -> u2
(u3, object_at_goal) -> u4
(u3, else) -> u3
REWARD_FUNCTION:
(u0, near_object & !object_grasped, u1) -> 0.1
(u1, object_grasped, u2) -> 0.15
(u2, object_near_goal | object_at_goal, u3) -> 0.15
(u3, object_at_goal, u4) -> 1

________________________________________________________________
               Meta-World Pick-Place — only after               

TASK: Place the puck at the goal only after moving it near the goal.

REWARD_MACHINE:
STATES: u0, u1, u2, u3
INITIAL_STATE: u0
FINAL_STATES: u3
TRANSITION_FUNCTION:
(u0, object_grasped) -> u1
(u0, else) -> u0
(u1, object_near_goal) -> u2
(u1, else) -> u1
(u2, object_at_goal) -> u3
(u2, else) -> u2
(u3, else) -> u3
REWARD_FUNCTION:
(u0, object_grasped, u1) -> 0.15
(u1, object_near_goal, u2) -> 0.35
(u2, object_at_goal, u3) -> 1

```